In [1]:
import requests
import pandas as pd
URL_TRIC = "https://parlement.tricoteuses.fr"
response=requests.get(URL_TRIC+"/v2/acteurs",
                       params={"perPage": 600, "chambre": "AN", "actif": "true"},
                       timeout=30)
deputes=pd.json_normalize(response.json()["data"])
deputes = deputes.rename(columns={
    "uid": "id",
    "groupeParlementaire.libelle": "groupe",
    "groupeParlementaire.libelleAbrege": "groupeAbrev",
    "circonscription.numCirco": "circo",
    "circonscription.numDepartement": "dep",
})
deputes=deputes.dropna(subset=["circo"])

deputes.shape[0]
# print(deputes.head())
print(deputes.columns)
deputes=deputes[["id","nom","prenom","groupe","groupeAbrev","circo","dep","compteBluesky"]]
deputes.head()
# deputes["uid"].nunique()

Index(['id', 'nom', 'prenom', 'civ', 'chambre', 'actif', 'urlImage',
       'interets', 'placeHemicycle', 'compteTwitter', 'compteBluesky',
       'adressesElectroniques', 'groupeParlementaire.uid', 'groupe',
       'groupeAbrev', 'groupeParlementaire.couleurAssociee',
       'groupeParlementaire.chambre', 'mandatPrincipal.uid',
       'mandatPrincipal.actif', 'mandatPrincipal.dateDebut',
       'mandatPrincipal.dateFin', 'mandatPrincipal.libQualite',
       'mandatPrincipal.codeQualite', 'mandatPrincipal.libelle',
       'circonscription.uid', 'circo', 'dep', 'circonscription.libelle',
       'commissionPermanente.uid', 'commissionPermanente.libelle',
       'commissionPermanente.libelleAbrege', 'groupeParlementaire',
       'circonscription', 'commissionPermanente'],
      dtype='str')


,id,nom,prenom,groupe,groupeAbrev,circo,dep,compteBluesky
0,PA1008,David,Alain,Socialistes et apparentés,SOC,4.0,33.0,alaindavid.bsky.social
2,PA1567,Guedj,Jérôme,Socialistes et apparentés,SOC,6.0,91.0,NaN
3,PA1592,Habib,David,"Libertés, Indépendants, Outre-mer et Territoires",LIOT,3.0,64.0,wrenswatch.bsky.social
4,PA1630,Herbillon,Michel,Droite Républicaine,DR,8.0,94.0,NaN
5,PA1654,Hollande,François,Socialistes et apparentés,SOC,1.0,19.0,fhollande.bsky.social


In [2]:
reponse = requests.get(URL_TRIC + "/adressesElectroniques/json",
                       params={"perPage": 10000}, timeout=30)
reponse.raise_for_status()

contacts = pd.json_normalize(reponse.json()["data"])
contacts = contacts.dropna(subset=["valElec"])
contacts = contacts.rename(columns={"acteurRefUid": "id", "valElec": "valeur"})
print(contacts["typeLibelle"].unique())
print(contacts.columns)

<ArrowStringArray>
[    'Téléphone',     'Télécopie',           'Mèl',  'Url sénateur',
 'Site internet',      'Facebook',       'Twitter',     'Instagram',
      'Linkedin']
Length: 9, dtype: str
Index(['uid', 'dataset', 'type', 'typeLibelle', 'poids',
       'adresseDeRattachement', 'valeur', 'xsiType', 'id', 'dateMaj',
       'datePremierAjout'],
      dtype='str')


In [3]:
contacts=contacts[["uid","id","typeLibelle","valeur"]]
contacts["valeur"] = contacts["valeur"].str.strip()
contacts=contacts.drop_duplicates(subset=["valeur"])
mask=(contacts["typeLibelle"] != "Mèl") | (
    (contacts["typeLibelle"] == "Mèl") & 
    (contacts["valeur"].str.endswith("assemblee-nationale.fr", na=False))
)
contacts = contacts[mask]
contacts.head()



,uid,id,typeLibelle,valeur
0,AD395657,PA1001,Téléphone,01 40 63 75 53
1,AD395658,PA1001,Télécopie,01 40 63 79 37
4,AD716102,PA1001,Mèl,mpdaubresse@assemblee-nationale.fr
5,AD736452,PA1001,Url sénateur,https://www.senat.fr/senateur/daubresse_marc_p...
6,AD191344,PA1002,Mèl,ydauge@assemblee-nationale.fr


In [4]:
contacts_pivot = contacts.pivot_table(
    index='id', 
    columns='typeLibelle', 
    values='valeur', 
    aggfunc=lambda x: '; '.join(x.astype(str))
).reset_index()

# Clean up the column header name
contacts_pivot.columns.name = None
contacts_pivot.head()

,id,Facebook,Instagram,Linkedin,Mèl,Site internet,Twitter,Télécopie,Téléphone,Url sénateur
0,PA1001,https://www.facebook.com/100057941235895,NaN,NaN,mpdaubresse@assemblee-nationale.fr,NaN,NaN,01 40 63 79 37,01 40 63 75 53,https://www.senat.fr/senateur/daubresse_marc_p...
1,PA1002,NaN,NaN,NaN,ydauge@assemblee-nationale.fr,NaN,NaN,NaN,NaN,https://www.senat.fr/senateur/dauge_yves01012h...
2,PA1008,NaN,NaN,NaN,Alain.David@assemblee-nationale.fr,alain-david.com,NaN,NaN,07 88 44 06 75,NaN
3,PA1009,NaN,NaN,NaN,mdavid@assemblee-nationale.fr,NaN,NaN,NaN,NaN,NaN
4,PA1012,charlesdelaverpilliere,NaN,NaN,charles.delaverpilliere@assemblee-nationale.fr,www.charlesdelaverpilliere.com,@delaverpilliere,NaN,04 74 35 13 58,NaN


In [5]:
depute_et_cont = deputes.merge(contacts_pivot, on='id', how='left',suffixes=('_depute', '_contact'))
depute_et_cont.columns
depute_et_cont=depute_et_cont.rename(columns={
    "Mèl":"E-mail officiel"
})
# depute_et_cont=depute_et_cont.drop[columns=["Url sénateur"]]
# depute_et_cont.tail()
depute_et_cont.loc[depute_et_cont["id"]=='PA795636']

,id,nom,prenom,groupe,groupeAbrev,circo,dep,compteBluesky,Facebook,Instagram,Linkedin,E-mail officiel,Site internet,Twitter,Télécopie,Téléphone,Url sénateur
381,PA795636,Lucas-Lundy,Benjamin,Écologiste et Social,EcoS,8.0,78.0,NaN,NaN,NaN,NaN,benjamin.lucas@assemblee-nationale.fr,benjaminlucas.fr,NaN,NaN,NaN,NaN


In [6]:
depute_et_cont.to_csv("deputes_et_contactes_tricouteuse.csv",index=False)